# 운수종사자 교통사고 위험 예측 데이터 탐색적 분석 (EDA)
---

## 1. 라이브러리 Import 및 설정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 시각화 설정
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

print("라이브러리 로드 완료!")

## 2. 데이터 로드

In [ ]:
# 데이터 로드
train_meta = pd.read_csv("./data/train.csv")
train_A = pd.read_csv("./data/train/A.csv")
train_B = pd.read_csv("./data/train/B.csv")

print(f"train.csv: {train_meta.shape}")
print(f"train/A.csv: {train_A.shape}")
print(f"train/B.csv: {train_B.shape}")

## 3. 기본 정보 분석

In [ ]:
# train.csv 기본 정보
print("[train.csv 기본 정보]")
print(train_meta.info())
print("\n[처음 5행]")
display(train_meta.head())

In [ ]:
# Label 분포
print("[Label 분포]")
print(train_meta['Label'].value_counts())
print(f"\n위험군(1) 비율: {train_meta['Label'].mean():.4f}")
print(f"정상군(0) 비율: {1-train_meta['Label'].mean():.4f}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label 분포 막대 그래프
train_meta['Label'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('Label Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Normal (0)', 'Risk (1)'], rotation=0)

# Label 비율 파이 차트
train_meta['Label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.2f%%', 
                                         labels=['Normal (0)', 'Risk (1)'],
                                         colors=['skyblue', 'salmon'])
axes[1].set_title('Label Proportion', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Test 종류별 분포
print("[Test 종류별 분포]")
print(train_meta['Test'].value_counts())

print("\n[Test 종류별 Label 분포]")
test_label_cross = pd.crosstab(train_meta['Test'], train_meta['Label'], normalize='index')
print(test_label_cross)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test 종류별 개수
train_meta['Test'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Test Type Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Test Type')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['A (New)', 'B (Maintain)'], rotation=0)

# Test 종류별 위험군 비율
test_label_cross.plot(kind='bar', stacked=False, ax=axes[1], color=['skyblue', 'salmon'])
axes[1].set_title('Risk Ratio by Test Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Test Type')
axes[1].set_ylabel('Proportion')
axes[1].set_xticklabels(['A (New)', 'B (Maintain)'], rotation=0)
axes[1].legend(['Normal (0)', 'Risk (1)'])

plt.tight_layout()
plt.show()

## 4. A 검사 (신규 자격) 상세 분석

In [ ]:
print(f"[A 검사 컬럼 수]: {len(train_A.columns)}")
print(f"\n[A 검사 컬럼 목록]:")
print(train_A.columns.tolist())

# Label과 병합
train_A_with_label = train_A.merge(train_meta[['Test_id', 'Label']], on='Test_id', how='left')

print(f"\n[A 검사 처음 3행]")
display(train_A_with_label.head(3))

In [ ]:
# A 검사 세부 항목별 컬럼 수
print("[A 검사 세부 항목별 컬럼 수]")
a_prefixes = {}
for col in train_A.columns:
    if col.startswith('A'):
        prefix = col.split('-')[0] if '-' in col else col
        a_prefixes[prefix] = a_prefixes.get(prefix, 0) + 1

a_prefix_series = pd.Series(a_prefixes).sort_index()
print(a_prefix_series)

# 시각화
plt.figure(figsize=(10, 5))
a_prefix_series.plot(kind='bar', color='steelblue')
plt.title('A Test - Number of Columns per Sub-test', fontsize=14, fontweight='bold')
plt.xlabel('Sub-test')
plt.ylabel('Number of Columns')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Age 분포 분석
print("[A 검사 Age 분포 Top 20]")
age_dist = train_A_with_label['Age'].value_counts().sort_index()
print(age_dist.head(20))

# Age별 위험군 비율
print("\n[A 검사 Age별 위험군 비율 Top 20]")
age_risk = train_A_with_label.groupby('Age')['Label'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(age_risk.head(20))

In [ ]:
# Age를 숫자로 변환하는 함수 (베이스라인 코드 참고)
def convert_age(val):
    if pd.isna(val): 
        return np.nan
    try:
        base = int(str(val)[:-1])
        return base if str(val)[-1] == "a" else base + 5
    except:
        return np.nan

train_A_with_label['Age_num'] = train_A_with_label['Age'].map(convert_age)

# Age 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Age 숫자 분포
train_A_with_label['Age_num'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('A Test - Age Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')

# Age별 위험군 비율 (count > 100인 경우만)
age_risk_filtered = age_risk[age_risk['count'] > 100].head(30)
age_risk_filtered['mean'].plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('A Test - Risk Ratio by Age (count > 100)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Risk Ratio')
axes[1].axhline(y=train_meta['Label'].mean(), color='red', linestyle='--', label='Overall Mean')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 결측치 분석
print("[A 검사 컬럼별 결측치 비율]")
null_ratio_A = (train_A.isnull().sum() / len(train_A) * 100).sort_values(ascending=False)
null_cols_A = null_ratio_A[null_ratio_A > 0]
print(null_cols_A.head(20))

if len(null_cols_A) > 0:
    plt.figure(figsize=(12, 6))
    null_cols_A.head(30).plot(kind='barh', color='steelblue')
    plt.title('A Test - Missing Value Ratio by Column', fontsize=14, fontweight='bold')
    plt.xlabel('Missing Ratio (%)')
    plt.ylabel('Column')
    plt.tight_layout()
    plt.show()
else:
    print("결측치가 없습니다!")

## 5. B 검사 (자격 유지) 상세 분석

In [ ]:
print(f"[B 검사 컬럼 수]: {len(train_B.columns)}")
print(f"\n[B 검사 컬럼 목록]:")
print(train_B.columns.tolist())

# Label과 병합
train_B_with_label = train_B.merge(train_meta[['Test_id', 'Label']], on='Test_id', how='left')

print(f"\n[B 검사 처음 3행]")
display(train_B_with_label.head(3))

In [ ]:
# B 검사 세부 항목별 컬럼 수
print("[B 검사 세부 항목별 컬럼 수]")
b_prefixes = {}
for col in train_B.columns:
    if col.startswith('B'):
        prefix = col.split('-')[0] if '-' in col else col
        b_prefixes[prefix] = b_prefixes.get(prefix, 0) + 1

b_prefix_series = pd.Series(b_prefixes).sort_index()
print(b_prefix_series)

# 시각화
plt.figure(figsize=(10, 5))
b_prefix_series.plot(kind='bar', color='coral')
plt.title('B Test - Number of Columns per Sub-test', fontsize=14, fontweight='bold')
plt.xlabel('Sub-test')
plt.ylabel('Number of Columns')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Age 분포 분석
train_B_with_label['Age_num'] = train_B_with_label['Age'].map(convert_age)

print("[B 검사 Age별 위험군 비율 Top 20]")
age_risk_b = train_B_with_label.groupby('Age')['Label'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(age_risk_b.head(20))

# Age 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Age 숫자 분포
train_B_with_label['Age_num'].hist(bins=30, ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('B Test - Age Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Frequency')

# Age별 위험군 비율 (count > 100인 경우만)
age_risk_b_filtered = age_risk_b[age_risk_b['count'] > 100].head(30)
age_risk_b_filtered['mean'].plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('B Test - Risk Ratio by Age (count > 100)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Risk Ratio')
axes[1].axhline(y=train_meta['Label'].mean(), color='red', linestyle='--', label='Overall Mean')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# 결측치 분석
print("[B 검사 컬럼별 결측치 비율]")
null_ratio_B = (train_B.isnull().sum() / len(train_B) * 100).sort_values(ascending=False)
null_cols_B = null_ratio_B[null_ratio_B > 0]
print(null_cols_B.head(20))

if len(null_cols_B) > 0:
    plt.figure(figsize=(12, 6))
    null_cols_B.head(30).plot(kind='barh', color='coral')
    plt.title('B Test - Missing Value Ratio by Column', fontsize=14, fontweight='bold')
    plt.xlabel('Missing Ratio (%)')
    plt.ylabel('Column')
    plt.tight_layout()
    plt.show()
else:
    print("결측치가 없습니다!")

## 6. 운수종사자 (PrimaryKey) 중복 검사 분석

In [ ]:
# A 검사: 동일 운수종사자가 여러 검사를 받은 경우
print("[A 검사: 동일 운수종사자가 여러 검사를 받은 경우]")
pk_counts_a = train_A_with_label.groupby('PrimaryKey').agg({
    'Test_id': 'count',
    'Label': 'mean'
}).rename(columns={'Test_id': 'test_count', 'Label': 'risk_ratio'})

duplicate_a = pk_counts_a[pk_counts_a['test_count'] > 1].sort_values('test_count', ascending=False)
print(duplicate_a.head(20))
print(f"\n중복 검사 운수종사자 수: {len(duplicate_a)}")
print(f"최대 검사 횟수: {pk_counts_a['test_count'].max()}")

In [ ]:
# B 검사: 동일 운수종사자가 여러 검사를 받은 경우
print("[B 검사: 동일 운수종사자가 여러 검사를 받은 경우]")
pk_counts_b = train_B_with_label.groupby('PrimaryKey').agg({
    'Test_id': 'count',
    'Label': 'mean'
}).rename(columns={'Test_id': 'test_count', 'Label': 'risk_ratio'})

duplicate_b = pk_counts_b[pk_counts_b['test_count'] > 1].sort_values('test_count', ascending=False)
print(duplicate_b.head(20))
print(f"\n중복 검사 운수종사자 수: {len(duplicate_b)}")
print(f"최대 검사 횟수: {pk_counts_b['test_count'].max()}")

In [ ]:
# 검사 횟수 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# A 검사
pk_counts_a['test_count'].value_counts().sort_index().head(10).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('A Test - Distribution of Test Count per Person', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Tests')
axes[0].set_ylabel('Number of People')
axes[0].tick_params(axis='x', rotation=0)

# B 검사
pk_counts_b['test_count'].value_counts().sort_index().head(10).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('B Test - Distribution of Test Count per Person', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Tests')
axes[1].set_ylabel('Number of People')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 7. 시퀀스 데이터 패턴 분석

In [ ]:
# A1-3 (반응 여부) 샘플 분석
print("[A1-3 반응 여부 시퀀스 샘플 분석]\n")
sample_a1_3 = train_A_with_label[['Test_id', 'A1-3', 'Label']].dropna().head(10)

for idx, row in sample_a1_3.iterrows():
    seq = str(row['A1-3']).split(',')
    resp_rate = seq.count('1') / len(seq) if len(seq) > 0 else 0
    print(f"Test_id: {row['Test_id']}, Label: {int(row['Label'])}, 반응률: {resp_rate:.3f}, 시퀀스 길이: {len(seq)}")

In [ ]:
# A1-4 (반응 시간) 샘플 분석
print("[A1-4 반응 시간 시퀀스 샘플 분석]\n")
sample_a1_4 = train_A_with_label[['Test_id', 'A1-4', 'Label']].dropna().head(10)

for idx, row in sample_a1_4.iterrows():
    try:
        seq = [float(x) for x in str(row['A1-4']).split(',')]
        mean_rt = np.mean(seq)
        std_rt = np.std(seq)
        print(f"Test_id: {row['Test_id']}, Label: {int(row['Label'])}, 평균 RT: {mean_rt:.2f}ms, 표준편차: {std_rt:.2f}ms, 길이: {len(seq)}")
    except:
        print(f"Test_id: {row['Test_id']} - 파싱 오류")

In [ ]:
# A1 반응률 피처 생성
def calculate_response_rate(series):
    return series.fillna("").apply(
        lambda x: str(x).split(",").count("1") / len(str(x).split(",")) if x else np.nan
    )

train_A_with_label['A1_resp_rate'] = calculate_response_rate(train_A_with_label['A1-3'])

print("[A1 반응률 기초 통계]")
print(train_A_with_label['A1_resp_rate'].describe())

In [ ]:
# A1 반응 시간 평균 피처 생성
def calculate_rt_mean(series):
    return series.fillna("").apply(
        lambda x: np.fromstring(str(x), sep=",").mean() if x else np.nan
    )

train_A_with_label['A1_rt_mean'] = calculate_rt_mean(train_A_with_label['A1-4'])

print("[A1 반응 시간 평균 기초 통계]")
print(train_A_with_label['A1_rt_mean'].describe())

## 8. 위험군 vs 정상군 비교 분석

In [ ]:
# A1 반응률: 위험군 vs 정상군 비교
risk_group = train_A_with_label[train_A_with_label['Label'] == 1]['A1_resp_rate'].dropna()
normal_group = train_A_with_label[train_A_with_label['Label'] == 0]['A1_resp_rate'].dropna()

print("[A1 반응률: 위험군 vs 정상군]")
print(f"위험군 평균: {risk_group.mean():.4f}")
print(f"정상군 평균: {normal_group.mean():.4f}")
print(f"차이: {risk_group.mean() - normal_group.mean():.4f}")

# t-test
if len(risk_group) > 0 and len(normal_group) > 0:
    t_stat, p_val = stats.ttest_ind(risk_group, normal_group, equal_var=False)
    print(f"\nt-statistic: {t_stat:.4f}")
    print(f"p-value: {p_val:.6f}")
    if p_val < 0.05:
        print("→ 통계적으로 유의미한 차이 존재 (p < 0.05)")
    else:
        print("→ 통계적으로 유의미한 차이 없음 (p >= 0.05)")

In [ ]:
# 시각화: A1 반응률 분포 비교
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 히스토그램
axes[0].hist(normal_group, bins=50, alpha=0.6, label='Normal (0)', color='skyblue', density=True)
axes[0].hist(risk_group, bins=50, alpha=0.6, label='Risk (1)', color='salmon', density=True)
axes[0].set_title('A1 Response Rate Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Response Rate')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 박스플롯
data_to_plot = [normal_group, risk_group]
axes[1].boxplot(data_to_plot, labels=['Normal (0)', 'Risk (1)'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('A1 Response Rate Boxplot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Response Rate')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# A1 반응 시간: 위험군 vs 정상군 비교
risk_rt = train_A_with_label[train_A_with_label['Label'] == 1]['A1_rt_mean'].dropna()
normal_rt = train_A_with_label[train_A_with_label['Label'] == 0]['A1_rt_mean'].dropna()

print("[A1 반응 시간: 위험군 vs 정상군]")
print(f"위험군 평균: {risk_rt.mean():.2f}ms")
print(f"정상군 평균: {normal_rt.mean():.2f}ms")
print(f"차이: {risk_rt.mean() - normal_rt.mean():.2f}ms")

# t-test
if len(risk_rt) > 0 and len(normal_rt) > 0:
    t_stat, p_val = stats.ttest_ind(risk_rt, normal_rt, equal_var=False)
    print(f"\nt-statistic: {t_stat:.4f}")
    print(f"p-value: {p_val:.6f}")
    if p_val < 0.05:
        print("→ 통계적으로 유의미한 차이 존재 (p < 0.05)")
    else:
        print("→ 통계적으로 유의미한 차이 없음 (p >= 0.05)")

In [ ]:
# 시각화: A1 반응 시간 분포 비교
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 히스토그램
axes[0].hist(normal_rt, bins=50, alpha=0.6, label='Normal (0)', color='skyblue', density=True)
axes[0].hist(risk_rt, bins=50, alpha=0.6, label='Risk (1)', color='salmon', density=True)
axes[0].set_title('A1 Response Time Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Response Time (ms)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 박스플롯
data_to_plot_rt = [normal_rt, risk_rt]
axes[1].boxplot(data_to_plot_rt, labels=['Normal (0)', 'Risk (1)'], patch_artist=True,
                boxprops=dict(facecolor='lightgreen'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_title('A1 Response Time Boxplot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Response Time (ms)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. TestDate (시간) 분석

In [ ]:
# TestDate를 년/월로 분리
def split_testdate(val):
    try:
        v = int(val)
        return v // 100, v % 100
    except:
        return np.nan, np.nan

train_A_with_label[['Year', 'Month']] = train_A_with_label['TestDate'].apply(
    lambda x: pd.Series(split_testdate(x))
)

# 년도별 검사 수
print("[A 검사 년도별 분포]")
print(train_A_with_label['Year'].value_counts().sort_index())

# 월별 검사 수
print("\n[A 검사 월별 분포]")
print(train_A_with_label['Month'].value_counts().sort_index())

In [ ]:
# 시각화: 시간에 따른 검사 수 추이
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 년도별
year_counts = train_A_with_label['Year'].value_counts().sort_index()
year_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('A Test - Count by Year', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# 월별
month_counts = train_A_with_label['Month'].value_counts().sort_index()
month_counts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('A Test - Count by Month', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 10. 상관관계 분석

In [ ]:
# 간단한 피처들과 Label의 상관관계
corr_features = ['Age_num', 'Year', 'Month', 'A1_resp_rate', 'A1_rt_mean', 'Label']
corr_data = train_A_with_label[corr_features].dropna()

print("[피처 간 상관관계 행렬]")
corr_matrix = corr_data.corr()
print(corr_matrix)

print("\n[Label과의 상관관계 (절대값 기준 정렬)]")
label_corr = corr_matrix['Label'].abs().sort_values(ascending=False)
print(label_corr)

In [ ]:
# 상관관계 히트맵
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. 분석 요약 및 인사이트

In [ ]:
print("=" * 80)
print("분석 요약 및 핵심 인사이트")
print("=" * 80)

print(f"""
1. 데이터 불균형
   - 위험군(Label=1) 비율: {train_meta['Label'].mean():.4f}
   - 정상군(Label=0) 비율: {1-train_meta['Label'].mean():.4f}
   → 클래스 불균형 처리 필요 (SMOTE, class_weight 등)

2. A 검사 vs B 검사
   - A 검사(신규): {len(train_A):,}개
   - B 검사(유지): {len(train_B):,}개
   → 각 검사마다 별도 모델 학습 권장

3. 운수종사자 중복 검사
   - A 검사 중복: {len(duplicate_a):,}명
   - B 검사 중복: {len(duplicate_b):,}명
   → 시계열 특성 활용 가능 (이전 검사 결과, 추세 등)

4. 시퀀스 데이터 특성
   - A 검사: {len([c for c in train_A.columns if c.startswith('A')])}개 컬럼
   - B 검사: {len([c for c in train_B.columns if c.startswith('B')])}개 컬럼
   → 반응률, 반응시간, 변동성 등 다양한 통계 피처 추출 가능

5. 인지적 특성 패턴
   - 좌/우 편향, 속도/정확성 트레이드오프
   - Stroop 효과, 주의 전환 능력 등
   → 베이스라인 코드의 피처 엔지니어링 참고

6. 결측치 패턴
   - 일부 검사 항목에서 결측치 존재
   → 결측 자체가 위험 신호일 수 있음

권장 모델링 접근:
✓ A/B 검사 별도 모델 학습 (베이스라인과 동일)
✓ 시퀀스 데이터에서 통계적 피처 추출 (평균, 표준편차, 비율, 차이 등)
✓ 동일 운수종사자의 이력 정보 활용
✓ 클래스 불균형 처리 (scale_pos_weight, SMOTE 등)
✓ LightGBM, XGBoost 등 트리 기반 모델 활용
✓ 교차 검증 및 앙상블 기법
""")

print("\n분석 완료! 🎉")